<a href="https://colab.research.google.com/github/ok124-droid/H-c-t-ng-c-ng/blob/main/CartPole.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Mô tả bài toán:
- Điều khiển một chiếc xe không có động cơ, trên một đường ray ma sát bằng 0 (lý tưởng hóa môi trường -> chỉ tập trung vào việc kiểm tả xem thuật toán thực sự tốt đến đâu)
- Trên xe có gắn một cây gậy, do trọng tâm nằm ở trên cao nên nó luôn có xu hướng đổ ập xuống do trọng lực
- Mục tiêu: đẩy xe sang trái, phải liên tục để giữ cho cây gậy luôn đứng thẳng lâu nhất có thể
- Các hành động có thể làm: đẩy xe sang trái, đẩy xe sang phải
- Không gian trạng thái:
  + Vị trí của xe: xe đang nằm ở đâu trên đường ray
  + Vận tốc của xe: xe đang lao về hướng nào, tốc độ bao nhiêu
  + Góc của cây gậy: gậy đang nghiêng về hướng nào, bao nhiêu độ
  + Vận tốc của cây gậy: gậy đang đổ xuống với vận tốc bao nhiêu
- Phần thưởng: cứ mỗi timestep mà gậy chưa đổ thì agent nhận được 1 điểm thưởng
- Điều kiện dừng: kết thúc ván nếu ít nhất 1 trong các điều kiện sau xảy ra:
  + Gậy đổ quá 12 độ
  + Xe đi quá tâm của đường ray 2.4 đơn vị
  + Hoàn thành hết 500 step


In [ ]:
import torch
import torch.nn as nn # nn là viết tắt của neport torch
import torch.nn as nn # nn là viết tắt của neural network cung cấp các layer, loss function
import torch.optim as optim # chứa các thuật toán tối ưu như SGD, Adam,..
import torch.nn.functional as F # cung cấp các hàm như ReLU, Softmax
from torch.distributions import Categorical # giúp agent chọn ngẫu nhiên hành động theo xác suất
import gymnasium as gym # cung cấp môi trường giả lập
import numpy as np

class PolicyNetwork(nn.Module):
  def __init__(self, input_dim, output_dim):
    # input_dim số lượng thông tin Agent nhình thấy (4)
    # output_dim số hành động agent có thể làm (2 - sang trái/ sang phải)
    super(PolicyNetwork, self).__init__()
    self.fc1 = nn.Linear(input_dim, 128)
    # lớp đầu vào nhận 4 thông số và nhân với ma trận trọng số để tạo ra biểu diễn mới gồm 128 đặc trưng ẩn
    self.fc2 = nn.Linear(128, output_dim)
    # lớp tiếp theo với đầu vào là 128 đặc trưng, đầu ra là điểm số cho 2 hành động sang trái và sang phải

  def forward(self, state):
    x = F.relu(self.fc1(state))
    logits = self.fc2(x)
    probs = F.softmax(logits, dim = -1)
    return probs


**Nếu làm cho PolicyNetwork trở thành một mạng phức tạp, nhiều lớp hơn liệu có làm cải thiện kết quả?**
- Thêm nhiều lớp ẩn: khi mạng lớn hơn, có nhiều lớp ẩn, khả năng học và ghi nhớ tốt hơn trong khi đầu vào chỉ có 4 đặc trưng -> mô hình học vẹt, ghi nhớ các nhiễu ngẫu nhiên của môi trường -> mất nhiều thời gian để hội tụ (khác với cnn khi đầu vào là ảnh dưới dạng các ma trận rất lớn thì mô hình cần phải nhiều lớp hơn để trích chọn đặc trưng, kết hợp các đặc trưng lại với nhau)
- Dùng dropout: sự ngẫu nhiên của mô trường và việc chọn hành động bằng cách bốc ngẫu nhiên theo xác suất đã quá đủ nhiễu rồi nếu thêm dropout thì sẽ bị quá nhiễu làm cho mô hình không thể học được một chính sách nào (giống như vừa xây lên được một chút thì lại đập đi sửa lại)

In [ ]:
def play_one_ep(env, policy):
  state, _ = env.reset() # khởi động lại, bắt đầu ván mới
  # theo công thức đạo hàm thì ta cần lưu lại log(xác suất hành động đã chọn) và điểm thưởng nhận được ở các bước
  log_probs = []
  rewards = []
  done = False

  while not done:
    state_tensor = torch.FloatTensor(state).unsqueeze(0)
    # môi trường gym trả về state dạng array
    # chuyển về tensor để phù hợp với mạng nơ ron
    # unsqueeze(0) đóng thành 1 batch do mạng nơ ron quen với việc xử lý theo batch (nhiều dữ liệu cùng lúc)
    probs = policy(state_tensor)
    m = Categorical(probs)
    action = m.sample()
    next_state, reward, terminated, truncated, _ = env.step(action.item())
    # termiated = True khi gậy đổ (nghiêng quá 12 độ) hoặc xe văng khỏi ranh giới
    # truncated = True nếu đủ 500 step
    done = terminated or truncated
    log_probs.append(m.log_prob(action))
    rewards.append(reward)
    state = next_state

  return log_probs, rewards

In [ ]:
def calculate_returns(rewards, gamma = 0.99):
  returns = []
  G = 0
  for r in reversed(rewards):
    G = r + gamma * G
    returns.insert(0, G)
  returns = torch.tensor(returns)
  # Chuẩn hóa
  # Vì hành động nào cũng có G > 0 nên phải chuẩn hóa để agent biết được hành động nào thực sự tốt, cần phải tăng xác suất
  # Giúp chuẩn hóa thang đo -> giảm exploding gradient đặc biệt khi G lớn
  returns = (returns - returns.mean()) / (returns.std() + 1e-9)
  return returns

In [ ]:
def train_reinforce():
  env = gym.make("CartPole-v1")
  policy = PolicyNetwork(env.observation_space.shape[0], env.action_space.n)
  optimizer = optim.Adam(policy.parameters(), lr = 1e-2)
  num_episodes = 500
  for episode in range(num_episodes):
    log_probs, rewards = play_one_ep(env, policy)
    returns = calculate_returns(rewards)
    policy_loss = []
    for log_prob, G in zip(log_probs, returns):
      policy_loss.append(-log_prob * G)
    policy_loss = torch.cat(policy_loss).sum()
    optimizer.zero_grad() # bắt đầu ván mới, đặt đạo hàm về lại 0 rồi mới tính
    policy_loss.backward()
    optimizer.step() # điều chỉnh lại tham số
    # Tuy trong công thức là cập nhật tham số theo Batch (chơi n ván mới cập nhật 1 lần)
    # Có thể là chơi 500 ván, cứ 5 ván cập nhật 1 lần chẳng hạn
    # Nhưng để tiết kiệm tài nguyên nên mình chọn chơi lần nào, cập nhật lần đó
    if episode % 50 == 0 or episode == (num_episodes - 1):
      total_reward = sum(rewards)
      print(f"Ván thứ {episode} - tổng điểm nhận được {total_reward}")
  env.close()

train_reinforce()


Ván thứ 0 - tổng điểm nhận được 15.0
Ván thứ 50 - tổng điểm nhận được 19.0
Ván thứ 100 - tổng điểm nhận được 500.0
Ván thứ 150 - tổng điểm nhận được 500.0
Ván thứ 200 - tổng điểm nhận được 450.0
Ván thứ 250 - tổng điểm nhận được 99.0
Ván thứ 300 - tổng điểm nhận được 500.0
Ván thứ 350 - tổng điểm nhận được 500.0
Ván thứ 400 - tổng điểm nhận được 500.0
Ván thứ 450 - tổng điểm nhận được 245.0
Ván thứ 499 - tổng điểm nhận được 288.0


- Việc điểm số thay đổi thất thường là do: agent đưa ra quyết định dựa theo xác suất, có thể chính sách(policy) đã rất tốt rồi, ví dụ mô hình đưa ra xác suất sang trái là 98% (mang lại kết quả tốt), sang phải 2% (mang lại kế quả xấu) thì nếu trong trường hợp hiếm hoi mà lỡ chọn sang phải dẫn đến mang lại kết quả xâu thì dẫn đến loss lớn, mô hình bị đánh giá là không thông minh trong khi thực tế nó đã làm rất tốt -> Vấn đề nằm ở chỗ là đánh giá thông qua 1 ván thì sẽ bị phiến diện làm cho điểm số bị giao động mạnh
- Ta nhận thấy ở ván 100 mô hình đã làm rất tốt (đạt 500 điểm) nhưng learning rate thì vẫn giữ nguyên điều này có thể làm mô hình đi lệch ra khỏi điểm tối ưu (đó cũng là lý do ra đời của nhiều kỹ thuật điều chỉnh learning rate như 1cycle, rồi các kỹ thuật dừng sớm, thuật toán Adam cũng là được dựa trên việc điều chỉnh tốc độ học kết hợp momentum)

**Actor-Critic**


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
import gymnasium as gym

class ActorCritic(nn.Module):
  def __init__(self, input_dim, n_actions):
    super(ActorCritic, self).__init__()
    self.fc1 = nn.Linear(input_dim, 128)
    self.fc2 = nn.Linear(128, 128)

    self.actor = nn.Linear(128, n_actions)
    self.critic = nn.Linear(128, 1)

  def forward(self, state):
    x = F.relu(self.fc1(state))
    x = F.relu(self.fc2(x))

    action_probs =F.softmax(self.actor(x), dim = -1)
    # critic đoán giá trị kỳ vọng tích lũy nhận được tính từ trạng thái hiện tại đến hết ván
    state_value = self.critic(x)
    return action_probs, state_value

In [ ]:
def train_actor_critic():
    env = gym.make('CartPole-v1')
    model = ActorCritic(env.observation_space.shape[0], env.action_space.n)

    # Chỉ cần 1 Huấn luyện viên cho cả 2 mạng (vì chúng chung cơ thể)
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    gamma = 0.99 # Hệ số chiết khấu tương lai

    num_episodes = 500

    for episode in range(num_episodes):
        state, _ = env.reset()
        done = False
        score = 0

        while not done:
            state_tensor = torch.FloatTensor(state)
            probs, state_val = model(state_tensor)

            m = Categorical(probs)
            action = m.sample()

            next_state, reward, terminated, truncated, _ = env.step(action.item())
            done = terminated or truncated
            score += reward

            next_state_tensor = torch.FloatTensor(next_state)
            _, next_state_val = model(next_state_tensor)

            if done:
                next_state_val = torch.tensor([0.0])

            td_target = reward + gamma * next_state_val
            delta = td_target - state_val

            critic_loss = F.mse_loss(state_val, td_target.detach())
            # tính lỗi của Actor = log_prob * delta (thay G_t bằng delta)
            actor_loss = -m.log_prob(action) * delta.detach()
            total_loss = critic_loss + actor_loss

            # cập nhật trọng số sau mỗi bước
            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()

            state = next_state

        if episode % 50 == 0:
            print(f"Ván {episode} | Điểm số (Sống sót): {score}")

    env.close()

train_actor_critic()

Ván 0 | Điểm số (Sống sót): 10.0
Ván 50 | Điểm số (Sống sót): 9.0
Ván 100 | Điểm số (Sống sót): 10.0
Ván 150 | Điểm số (Sống sót): 9.0
Ván 200 | Điểm số (Sống sót): 9.0
Ván 250 | Điểm số (Sống sót): 10.0
Ván 300 | Điểm số (Sống sót): 10.0
Ván 350 | Điểm số (Sống sót): 8.0
Ván 400 | Điểm số (Sống sót): 9.0
Ván 450 | Điểm số (Sống sót): 10.0


**Tại sao actor_critic lại cho ra kết quả rất tệ**
- Cập nhật ở mỗi bước bằng cách, tính delta = phần thưởng nhận được + giá trị của trạng thái tiếp theo - giá trị của trạng thái ban đầu, có nghĩa là ta cho rằng giá trị của trạng thái đang bị dự đoán sai delta đơn vị, nhưng thực tế là ban đầu model chưa biết gì cả mọi giá trị trạng thái nó đưa ra đều chỉ là các con số ngẫu nhiên nên delta thực sự không phản ánh được gì (khác với học có giám sát, bạn có nhãn chính xác và bạn dùng nhãn này làm thước đo để cập nhật trọng số thực sự giúp cải thiện mô hình vì nó là thước đo chuẩn). Cộng thêm việc cập nhật trọng số bằng việc đánh giá ở 1 bước duy nhất là quá tự tin (bias cao), kết quả là do điều chỉnh trọng số sai hướng -> tăng xác suất thực hiện hành động sai -> agent nhanh chết -> không kịp khám phá nhiều trạng thái mới -> critic không học được -> tiếp tục cập nhật sai

**Actor critic kết hợp với montecalo có tốt hơn không?**
- Lần này ta tính delta = G_t - V_t có nghĩa là điểm số thực tế nhận được từ thời điểm t đến khi hết ván trừ đi điểm số dự đoán, nhằm cung cấp đáp án chính xác để cập nhật trọng số tránh việc cập nhật sai do quá tự tin; tuy nhiên cách này đánh giá trên một chuỗi hành động dài nên sẽ rất nhiễu vì hành động được chọn dựa vào việc bốc ngẫu nhiên theo phân phối xác suất ví dụ như việc hành động ở bước thứ 5 có thể không ảnh hưởng mấy đến kết quả tệ ở bước 20 mà là do một sự ngẫu nhiên ở bước 18 nhưng G_t vẫn vì sự kiện này mà thay đổi, tất nhiên điều này cũng đã được giảm bướt phần nào nhờ hệ số gamma
- Khi kết hợp với montecalo làm cho agent mất khả năng học nhanh vì phải chờ ván chơi kết thúc mới có thể cập nhật được và phương pháp này là không khả thi nếu ván chơi hoặc môi trường là không gian vô tận



In [ ]:
def play_ten_episodes(env, model, is_frozen, gamma = 0.99, batch_size = 10):
  batch_log_probs = []
  batch_values = []
  batch_returns = []
  batch_scores = []

  for episode in range(batch_size):
    state, _ = env.reset()
    done = False
    ep_rewards = []
    ep_log_probs = []
    ep_values = []
    score = 0
    while not done:
      state_tensor = torch.FloatTensor(state)
      probs, state_val = model(state_tensor)

      m = Categorical(probs)

      if is_frozen:
        action = torch.argmax(probs)
      else:
        action = m.sample()

      next_state, reward, terminated, truncated, _ = env.step(action.item())
      done = terminated or truncated

      if not terminated: score += 1 # nếu không đổ thì +1 điểm
      if terminated: reward = -10.0 # nếu làm đổ phạt mạnh 10

      ep_log_probs.append(m.log_prob(action))
      ep_values.append(state_val)
      ep_rewards.append(reward)

      state = next_state

    batch_scores.append(score)
    ep_returns = []
    G = 0
    for r in reversed(ep_rewards):
      G = r + gamma * G
      ep_returns.insert(0, G)

    batch_returns.extend(ep_returns)
    batch_log_probs.extend(ep_log_probs)
    batch_values.extend(ep_values)

  return batch_log_probs, batch_values, batch_returns, batch_scores



**Bằng các kiến thức hiện tại của mình, tôi đưa ra giải pháp là**:
- Hi sinh khả năng học nhanh của actor critic, và giảm bớt nhiễu của monte carlo tôi sẽ cho agent chơi 10 ván sau đó mới cập nhật trọng số một lần nhằm giảm bớt các yếu tố ngẫu nhiên của môi trường và việc chọn hành động theo xác suất
- Khi agent đã đạt điểm tối đa (500) điểm tôi mặc định đây là chính sách tốt và quyết định ngừng cập nhật trọng số và ở mỗi bước sẽ chọn hành động có xác suất cao nhất để tránh sự ngẫu nhiên của việc chọn hành động theo phân phối xác suất vô tình làm đánh giá sai một chính sách tốt
- Tiếp tục chơi đến khi nào điểm số trung bình của 10 ván nhỏ hơn một mốc cho sẵn (ví dụ 400) thì tiếp tục cập nhật trọng số trở lại, đây là cách mà tôi cho agent thích nghi với việc môi trường thay đổi.

In [ ]:
def train_modular_actor_critic():
    env = gym.make('CartPole-v1')
    model = ActorCritic(env.observation_space.shape[0], env.action_space.n)
    optimizer = optim.Adam(model.parameters(), lr=1e-2)

    num_iterations = 100
    is_frozen = False

    for iteration in range(num_iterations):
        b_log_probs, b_values, b_returns, b_scores = play_ten_episodes(env, model, is_frozen)
        avg_score = sum(b_scores) / len(b_scores)

        # Cập nhật trọng số nếu không đóng băng
        if not is_frozen:
            b_returns = torch.tensor(b_returns)
            b_log_probs = torch.stack(b_log_probs)
            b_values = torch.cat(b_values).squeeze()

            # Chuẩn hóa và tính lỗi
            b_returns = (b_returns - b_returns.mean()) / (b_returns.std() + 1e-9)
            advantage = b_returns - b_values.detach()

            actor_loss = -(b_log_probs * advantage).mean()
            critic_loss = F.mse_loss(b_values, b_returns)
            total_loss = actor_loss + critic_loss

            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()

            print(f"Đang học | lần cập nhật {iteration+1:03d} | điểm TB 10 ván: {avg_score:.1f}")
        else:
            print(f"Đang đóng băng | điểm TB 10 ván: {avg_score:.1f}")

        # Kiểm tra xem nên đóng băng không
        if not is_frozen and avg_score == 500.0:
            is_frozen = True
            print("ĐÃ ĐÓNG BĂNG!")

        elif is_frozen and avg_score < 400.0:
            is_frozen = False
            print("ĐÃ RÃ ĐÔNG!")

    env.close()

In [ ]:
train_modular_actor_critic()

Đang học | lần cập nhật 001 | điểm TB 10 ván: 25.2
Đang học | lần cập nhật 002 | điểm TB 10 ván: 14.0
Đang học | lần cập nhật 003 | điểm TB 10 ván: 27.6
Đang học | lần cập nhật 004 | điểm TB 10 ván: 30.3
Đang học | lần cập nhật 005 | điểm TB 10 ván: 23.7
Đang học | lần cập nhật 006 | điểm TB 10 ván: 39.6
Đang học | lần cập nhật 007 | điểm TB 10 ván: 45.3
Đang học | lần cập nhật 008 | điểm TB 10 ván: 52.8
Đang học | lần cập nhật 009 | điểm TB 10 ván: 48.2
Đang học | lần cập nhật 010 | điểm TB 10 ván: 67.4
Đang học | lần cập nhật 011 | điểm TB 10 ván: 80.7
Đang học | lần cập nhật 012 | điểm TB 10 ván: 84.4
Đang học | lần cập nhật 013 | điểm TB 10 ván: 114.4
Đang học | lần cập nhật 014 | điểm TB 10 ván: 85.8
Đang học | lần cập nhật 015 | điểm TB 10 ván: 128.7
Đang học | lần cập nhật 016 | điểm TB 10 ván: 134.9
Đang học | lần cập nhật 017 | điểm TB 10 ván: 103.2
Đang học | lần cập nhật 018 | điểm TB 10 ván: 130.4
Đang học | lần cập nhật 019 | điểm TB 10 ván: 167.8
Đang học | lần cập nhật 0

- Ta thấy sau khi đóng băng thì chỉ có 1 ván không đạt điểm tối đa (498.2 điểm) còn lại tất cả các ván đều đạt 500 điểm chứng tỏ chính sách hiện tại đã rất tốt.